### Data setup, one-time instanciation of SQL database

Raw dataset: https://www.kaggle.com/datasets/tatheerabbas/industrial-machine-predictive-maintenance

In [1]:
import duckdb
import pandas as pd

In [2]:
# open file
df = pd.read_csv('../data/raw/predictive_maintenance_v3.csv')

# fix timestamp date type
df['timestamp'] = pd.to_datetime(df['timestamp'])

# create entity column
entity_group_dict =  {'CNC': 'CNC',
                     'Pump': 'PMP',
                     'Compressor': 'CPR',
                     'Robotic Arm': 'ARM'
                     }

df['entity_group'] = df['machine_type'].replace(entity_group_dict)
df['entity'] = df['entity_group'] + df['machine_id'].astype('str').str.zfill(2)

df.head()


,timestamp,machine_id,machine_type,vibration_rms,temperature_motor,current_phase_avg,pressure_level,rpm,operating_mode,hours_since_maintenance,ambient_temp,rul_hours,failure_within_24h,failure_type,estimated_repair_cost,entity_group,entity
0,2024-01-01 00:00:00,1,CNC,0.81,49.51,5.10,23.6,860.9,idle,273.80,13.9,61.00,0,none,0,CNC,CNC01
1,2024-01-01 00:03:00,1,CNC,0.75,40.58,5.30,23.6,899.6,idle,273.85,10.2,60.95,0,none,0,CNC,CNC01
2,2024-01-01 00:21:00,1,CNC,0.71,49.70,NaN,21.3,862.7,idle,274.15,13.6,60.65,0,none,0,CNC,CNC01
3,2024-01-01 00:45:00,1,CNC,0.76,43.04,4.79,22.6,870.4,idle,274.55,13.4,60.25,0,none,0,CNC,CNC01
4,2024-01-01 00:54:00,1,CNC,0.88,41.39,4.44,22.2,881.9,idle,274.70,10.8,60.10,0,none,0,CNC,CNC01


In [3]:
# convert table to stacked using melt

df = df.sort_values('timestamp')

# Convert table to stacked
SENSOR_COLUMNS = [
    'vibration_rms',
    'temperature_motor',
    'current_phase_avg',
    'pressure_level',
    'rpm',
    'ambient_temp'
]

df_long = df.melt(
    id_vars=[
        'timestamp',
        'entity',
        'entity_group',
        'operating_mode',
        'hours_since_maintenance',
        'failure_within_24h',
        'rul_hours',
        'failure_type'
    ],
    value_vars=SENSOR_COLUMNS,
    var_name='sensor',
    value_name='value'
)

# Drop missing sensor readings
df_long = df_long.dropna(subset=['value'])

# Save table to processed
df_long.to_csv('../data/processed/predictive_maintenance.csv', index=False)

df_long.head()

,timestamp,entity,entity_group,operating_mode,hours_since_maintenance,failure_within_24h,rul_hours,failure_type,sensor,value
0,2024-01-01,CNC01,CNC,idle,273.80,0,61.00,none,vibration_rms,0.81
1,2024-01-01,CNC03,CNC,idle,158.24,0,98.34,none,vibration_rms,0.78
2,2024-01-01,ARM19,ARM,idle,72.78,0,73.90,none,vibration_rms,0.44
3,2024-01-01,CNC04,CNC,idle,140.73,0,87.91,none,vibration_rms,0.72
4,2024-01-01,ARM18,ARM,idle,206.73,0,92.86,none,vibration_rms,0.44


In [4]:
# generate chart statistics and SPC limits

# Filter operation mode = normal
df_long = df_long[df_long['operating_mode'] == 'normal']

spc_stats = (
    df_long
    .groupby(['entity_group', 'sensor'], as_index=False)
    .agg(
        mean=('value', 'mean'),
        std=('value', 'std')
    )
)

spc_stats['ucl'] = spc_stats['mean'] + 3 * spc_stats['std']
spc_stats['centerline'] = spc_stats['mean']
spc_stats['lcl'] = spc_stats['mean'] - 3 * spc_stats['std']

# Save table to processed
spc_stats.to_csv('../data/processed/chart_limits.csv', index=False)

spc_stats

,entity_group,sensor,mean,std,ucl,centerline,lcl
0,ARM,ambient_temp,13.037045,2.935995,21.845029,13.037045,4.229061
1,ARM,current_phase_avg,5.823174,0.620497,7.684665,5.823174,3.961682
2,ARM,pressure_level,42.908369,3.515778,53.455703,42.908369,32.361035
3,ARM,rpm,349.414256,14.877368,394.046361,349.414256,304.782151
4,ARM,temperature_motor,44.605636,7.930329,68.396624,44.605636,20.814648
5,ARM,vibration_rms,1.233301,0.340565,2.254997,1.233301,0.211606
6,CNC,ambient_temp,12.962311,2.885127,21.617691,12.962311,4.306931
7,CNC,current_phase_avg,13.264640,2.337649,20.277587,13.264640,6.251693
8,CNC,pressure_level,71.167044,11.154129,104.629431,71.167044,37.704657
9,CNC,rpm,2911.171285,119.268715,3268.977431,2911.171285,2553.365139


In [5]:
# create sql instance, load processed CSVs
con = duckdb.connect('../data/mfg.duckdb')

con.execute("""
CREATE TABLE IF NOT EXISTS sensor_data AS
SELECT *
FROM read_csv_auto('../data/processed/predictive_maintenance.csv')
""")

con.execute("""
CREATE TABLE IF NOT EXISTS sensor_spc_limits AS
SELECT *
FROM read_csv_auto('../data/processed/chart_limits.csv')
""")

# duckdb close connection doesn't seem to work. shut down kernel when finished to end process lock.
con.close()